# M8: Static Batching

Up to M7 the scheduler ran one request at a time. Every step read all of the model's weights to produce a single token. M8 runs up to `max_batch_size` requests in **one** forward pass.

```text
get_next_batch_to_run   running_batch empty? take up to max_batch_size from waiting_queue
        │               otherwise keep running the current batch (no refill)
        ▼
ModelRunner.forward     prepare_inputs(batch) → input_ids / attention_mask / position_ids  [B, T]
        │               self.model(...) → logits [B, T, vocab] → logits[:, -1]  [B, vocab]
        ▼
Sampler                 argmax(dim=-1) → one token per request
        ▼
process_batch_result    result[i] belongs to batch[i]; end_req() for the ones that finished
```

The four-step scheduler loop, streaming, and abort are unchanged. What changes is that a "batch" is now a list of requests, and the model needs a rectangular tensor.

## Setup

No HTTP server this time. `Engine` loads the tokenizer and the model; the experiments below use `ModelRunner` directly, or a fresh `Scheduler` whose requests are all queued before its thread starts, so the first step sees every one of them.

In [1]:
import threading
import time

import torch

from engine import Engine
from req import Req
from scheduler import Scheduler

engine = Engine()
runner = engine.model_runner
tok = engine.tokenizer
print(runner.device)


def make_req(rid: str, prompt: str, max_new_tokens: int, eos: bool = True) -> Req:
    # eos=False: ignore EOS, so the request runs exactly max_new_tokens steps.
    return Req(
        rid=rid,
        prompt=prompt,
        input_ids=tok.encode(prompt),
        max_new_tokens=max_new_tokens,
        eos_token_ids=runner.eos_token_ids if eos else set(),
    )


def run_static(reqs: list[Req], max_batch_size: int) -> list[list[str]]:
    # Queue every request first, then start the loop. Returns the rids in each step's batch.
    scheduler = Scheduler(runner, max_batch_size=max_batch_size)
    steps = []
    run_batch = scheduler.run_batch

    def logged(batch):
        steps.append([req.rid for req in batch])
        return run_batch(batch)

    scheduler.run_batch = logged
    for req in reqs:
        scheduler.submit(req)
    threading.Thread(target=scheduler.event_loop, daemon=True).start()
    for req in reqs:
        req.done.wait()
    return steps

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps


## One Forward, Many Requests

Three requests, `max_batch_size=8`. Each step's batch holds all three, so each step is one forward pass that gives every request one token.

In [2]:
reqs = [make_req(rid, prompt, 5) for rid, prompt in [("A", "The capital of France is"), ("B", "Hi"), ("C", "One two three")]]
steps = run_static(reqs, max_batch_size=8)

for i, rids in enumerate(steps, 1):
    print(f"step {i}: {rids}")
for req in reqs:
    print(f"{req.rid}: {req.status.name:<8} {req.finished_reason.to_json()}  {tok.decode(req.output_ids)!r}")

step 1: ['A', 'B', 'C']
step 2: ['A', 'B', 'C']
step 3: ['A', 'B', 'C']
step 4: ['A', 'B', 'C']
step 5: ['A', 'B', 'C']
A: FINISHED {'type': 'length', 'length': 5}  ' Paris. The capital of'
B: FINISHED {'type': 'length', 'length': 5}  'Question\nOkay, so'
C: FINISHED {'type': 'length', 'length': 5}  ', two two two,'


## The Padded Layout

The model takes `[batch_size, seq_len]`, and every row must have the same length. `prepare_inputs` left-pads the shorter sequences:

- `input_ids`: pad slots hold `PAD_TOKEN_ID` (0, which happens to be `!`). Any id works, because the mask hides it.
- `attention_mask`: 1 for real tokens, 0 for padding.
- `position_ids`: each row counts from its own first real token, the same positions it has when run alone.

Left padding puts every row's last real token in the last column, so `logits[:, -1]` is right for every row.

In [3]:
batch = [make_req("A", "The capital of France is", 4), make_req("B", "Hi", 4), make_req("C", "One two three", 4)]
input_ids, attention_mask, position_ids = runner.prepare_inputs(batch)

print("shape:", tuple(input_ids.shape), input_ids.dtype, input_ids.device)
for name, t in [("input_ids", input_ids), ("attention_mask", attention_mask), ("position_ids", position_ids)]:
    print(f"\n{name}")
    for req, row in zip(batch, t.tolist(), strict=True):
        print(f"  {req.rid}  {row}")

shape: (3, 5) torch.int64 mps:0

input_ids
  A  [785, 6722, 315, 9625, 374]
  B  [0, 0, 0, 0, 13048]
  C  [0, 0, 3966, 1378, 2326]

attention_mask
  A  [1, 1, 1, 1, 1]
  B  [0, 0, 0, 0, 1]
  C  [0, 0, 1, 1, 1]

position_ids
  A  [0, 1, 2, 3, 4]
  B  [0, 0, 0, 0, 0]
  C  [0, 0, 0, 1, 2]


## What the Mask and `position_ids` Each Fix

Call the model three ways on one padded batch and compare each row with running that request alone. A is long, so B and C get a lot of padding. A itself has no padding: its difference is only rounding noise from a different batch shape, and serves as the noise floor.

In [4]:
batch = [
    make_req("A", "In the beginning of the story, a young dragon lived alone on a quiet mountain far away from every village, and each night it watched", 1),
    make_req("B", "Hi", 1),
    make_req("C", "The capital of France is", 1),
]
input_ids, attention_mask, position_ids = runner.prepare_inputs(batch)
print("padding per row:", [int((m == 0).sum()) for m in attention_mask], "\n")

with torch.inference_mode():
    alone = [runner.forward([req])[0].float() for req in batch]
    variants = {
        "mask + position_ids": {"attention_mask": attention_mask, "position_ids": position_ids},
        "mask only": {"attention_mask": attention_mask},
        "neither": {},
    }
    for name, kwargs in variants.items():
        logits = runner.model(input_ids, use_cache=False, **kwargs).logits[:, -1].float()
        diffs = "  ".join(f"{req.rid}={(row - a).abs().max().item():6.3f}" for req, row, a in zip(batch, logits, alone))
        same = [req.rid for req, row, a in zip(batch, logits, alone) if row.argmax() == a.argmax()]
        print(f"{name:<20} max |diff|: {diffs}   same argmax: {same}")

padding per row: [0, 26, 22] 



mask + position_ids  max |diff|: A= 0.250  B= 0.062  C= 0.387   same argmax: ['A', 'B', 'C']
mask only            max |diff|: A= 0.250  B= 0.062  C= 0.281   same argmax: ['A', 'B', 'C']


neither              max |diff|: A= 0.250  B=13.656  C= 6.844   same argmax: ['A', 'C']


- **neither**: real tokens attend to the padding as if it were text. B and C change a lot, and the argmax can flip. The mask is what makes batching correct.
- **mask only**: B and C stay at the noise floor. Without `position_ids` the model numbers positions by column, so every real token in a padded row is shifted by the same amount. RoPE only encodes the distance between two tokens, and a uniform shift leaves every distance the same.
- **mask + position_ids**: each row gets exactly the positions it has when run alone.

So for Qwen3, `position_ids` is about matching the unbatched computation exactly, not about a large error. It is still required: a model with learned absolute position embeddings, such as GPT-2, adds a different vector for each position, and a shifted row would get the wrong ones. Any difference left in the first variant is rounding: a different batch shape can make bf16 kernels round a little differently, on CPU as well as MPS or CUDA.

## Same Tokens as Running Alone

The contract of M8: batching must not change any request's output. Run each prompt alone (`max_batch_size=1`), then all together, and compare greedy tokens.

In [5]:
prompts = {
    "A": "The capital of France is",
    "B": "Hi",
    "C": "Write a haiku about the sea.",
    "D": "1, 2, 3, 4,",
}

alone = {}
for rid, prompt in prompts.items():
    req = make_req(rid, prompt, 12)
    run_static([req], max_batch_size=1)
    alone[rid] = req.output_ids

together = [make_req(rid, prompt, 12) for rid, prompt in prompts.items()]
run_static(together, max_batch_size=8)

for req in together:
    print(f"{req.rid}: same={req.output_ids == alone[req.rid]}  {tok.decode(req.output_ids)!r}")

A: same=True  ' Paris. The capital of France is also the capital of the'
B: same=True  'Question\nOkay, so I need to figure out how to'
C: same=True  ' The sea is a place where people come to live, and'
D: same=True  ' 5, 6, 7, 8,'


If a row prints `same=False` on MPS or CUDA, it is the bf16 rounding from the previous section: a near-tie between two tokens flipped, and greedy decoding follows a different path from there. The tests check this on CPU.

## Why Batching Raises Throughput

Time one forward step for batch sizes 1 to 32, with the same prompt in every row. GPU work is asynchronous, so synchronize before reading the clock.

Each step reads all the weights once, whatever the batch size. While the device is waiting on memory, extra rows are almost free: the step time grows slowly and tokens per second climbs. Once the device is busy computing, the step time grows with the batch and tokens per second levels off.

In [6]:
def sync() -> None:
    if runner.device == "cuda":
        torch.cuda.synchronize()
    elif runner.device == "mps":
        torch.mps.synchronize()


prompt_ids = tok.encode("Write a short story about a dragon who learns to fly.")
print(f"prompt length: {len(prompt_ids)} tokens\n")
base = None
for bs in [1, 2, 4, 8, 16, 32]:
    rows = [Req(rid=str(i), prompt="", input_ids=prompt_ids, max_new_tokens=1) for i in range(bs)]
    runner.forward(rows)  # warm up this shape
    sync()
    start = time.perf_counter()
    for _ in range(5):
        runner.forward(rows)
    sync()
    step = (time.perf_counter() - start) / 5
    base = base or bs / step
    print(f"B={bs:>2}  step {step * 1000:7.1f} ms   tokens/s {bs / step:8.1f}   x{bs / step / base:5.1f} vs B=1")

prompt length: 12 tokens



B= 1  step    42.6 ms   tokens/s     23.5   x  1.0 vs B=1


B= 2  step    58.2 ms   tokens/s     34.3   x  1.5 vs B=1


B= 4  step    93.2 ms   tokens/s     42.9   x  1.8 vs B=1


B= 8  step   132.2 ms   tokens/s     60.5   x  2.6 vs B=1


B=16  step   215.4 ms   tokens/s     74.3   x  3.2 vs B=1


B=32  step   391.1 ms   tokens/s     81.8   x  3.5 vs B=1


There is no KV cache yet, so each row is the whole prompt, not one token. A single request already has many rows, and the gain is smaller than it will be once decoding feeds one token per request.

## Static Batching Leaves Slots Empty

Four requests with different lengths (EOS ignored, so each runs exactly `max_new_tokens` steps), `max_batch_size=3`. A, B, C start together. A and C finish early, but their slots stay empty until B is done, and D waits the whole time.

In [7]:
lengths = {"A": 2, "B": 10, "C": 3, "D": 4}
reqs = [make_req(rid, "Tell me a story.", n, eos=False) for rid, n in lengths.items()]
steps = run_static(reqs, max_batch_size=3)

for i, rids in enumerate(steps, 1):
    print(f"step {i:>2}: {' '.join(rids) + ' ·' * (3 - len(rids)):<8}")
used = sum(len(rids) for rids in steps)
print(f"\nslots used: {used} / {len(steps) * 3} ({used / (len(steps) * 3):.0%})")
print(f"D started at step {next(i for i, rids in enumerate(steps, 1) if 'D' in rids)}, though a slot was free from step 3")

step  1: A B C   
step  2: A B C   
step  3: B C ·   
step  4: B · ·   
step  5: B · ·   
step  6: B · ·   
step  7: B · ·   
step  8: B · ·   
step  9: B · ·   
step 10: B · ·   
step 11: D · ·   
step 12: D · ·   
step 13: D · ·   
step 14: D · ·   

slots used: 19 / 42 (45%)
D started at step 11, though a slot was free from step 3


## One Failure Fails the Batch

A forward pass belongs to the whole batch, so when it raises, every request in that batch ends as `FAILED`. The scheduler thread keeps running and serves the next batch.

In [8]:
def failing_forward(batch):
    raise RuntimeError("model crashed")


runner.forward = failing_forward
reqs = [make_req(rid, "Hello", 5) for rid in "AB"]
run_static(reqs, max_batch_size=8)
for req in reqs:
    print(req.rid, req.status.name, req.finished_reason.to_json())
del runner.forward  # back to the real model

later = make_req("C", "Hello", 3)
run_static([later], max_batch_size=8)
print(later.rid, later.status.name, repr(tok.decode(later.output_ids)))

A FAILED {'type': 'error', 'message': 'model crashed'}
B FAILED {'type': 'error', 'message': 'model crashed'}
C FINISHED ' Answer! I'


**Something to think about.** HF returns logits for every position, `[B, T, vocab]`, and we keep only the last column. With `vocab = 151936`, a batch of 8 prompts of 500 tokens holds about 1.2 GB of logits in bf16 that are thrown away. Look up `logits_to_keep` in the model's `forward` docstring.

## Summary

| | M7 | M8 |
|---|---|---|
| Unit of work | one `Req` | `list[Req]`, up to `max_batch_size` |
| Scheduler state | `running_req` | `running_batch`; `end_req` removes finished requests |
| Admission | next waiting request when idle | a new batch only when `running_batch` is empty; free slots are not refilled |
| Model input | `[1, T]` | left-padded `[B, T]`; `attention_mask` hides padding, `position_ids` match each row run alone |
| Model output | `logits[0, -1]` → one token | `logits[:, -1]` → `[B, vocab]` → one token per row |
| Error in forward | that request fails | every request in the batch fails |
| Output | — | token-identical to running each request alone |

Batching raises throughput because one weight read serves many rows. Static batching wastes the slots of requests that finish early; refilling them is the next step.

**Compare with SGLang**: `get_new_batch_prefill()` and `get_num_allocatable_reqs()` in `managers/scheduler.py` (the `--max-running-requests` cap). SGLang does not pad: it packs tokens and finds each request's last token with `last_index` in `layers/logits_processor.py`.